# 02 - Entrenamiento de Modelos Prophet

Entrenamiento y evaluación de modelos de series temporales para predicción energética.

**Modelos:** 20 (4 sedes x 5 sectores)
**Métricas objetivo:** MAE < 15 kWh | R² > 0.85 | MAPE < 20%

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import pickle
import os
import sys

sys.path.insert(0, '../backend')
from api.config import SEDES, SECTORES, SECTOR_COLUMNS

In [ ]:
df = pd.read_csv('../backend/data/consumos_uptc.csv', parse_dates=['timestamp'])
print(f'Registros: {len(df):,}')

In [ ]:
# Ejemplo: entrenar modelo Tunja/Comedores
sede = 'Tunja'
sector = 'Comedores'
col = SECTOR_COLUMNS[sector]

df_sede = df[df['sede'] == sede].copy()
df_prophet = pd.DataFrame({
    'ds': df_sede['timestamp'],
    'y': df_sede[col].astype(float),
    'temperatura_exterior_c': df_sede['temperatura_exterior_c'].astype(float),
    'ocupacion_pct': df_sede['ocupacion_pct'].astype(float),
    'es_fin_semana': df_sede['es_fin_semana'].astype(int),
}).dropna()

print(f'Registros para entrenamiento: {len(df_prophet)}')

In [ ]:
# Train/test split temporal
cutoff = df_prophet['ds'].max() - pd.Timedelta(days=30)
train = df_prophet[df_prophet['ds'] <= cutoff]
test = df_prophet[df_prophet['ds'] > cutoff]
print(f'Train: {len(train)} | Test: {len(test)}')

In [ ]:
# Entrenar modelo
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    interval_width=0.95,
    growth='linear'
)
model.add_regressor('temperatura_exterior_c')
model.add_regressor('ocupacion_pct')
model.add_regressor('es_fin_semana')

model.fit(train)

In [ ]:
# TODO: Evaluar métricas MAE, R², MAPE
# TODO: Visualizar predicciones vs reales
# TODO: Cross-validation con Prophet
# TODO: Entrenar todos los 20 modelos
# TODO: Guardar modelos en models/trained/